# NDF USD/COP — Valoración Detallada

Análisis completo de un Non-Deliverable Forward confirmado.

---

## Términos del Trade

| Campo | Valor |
|---|---|
| Ticker | **FW-BOCS-05.02.2026** |
| Contraparte | Banco de Occidente |
| Tipo | Non-Deliverable Forward (NDF) |
| Nocional | **USD 500,000** |
| Strike | **3,717.75 USD/COP** |
| Fecha de inicio | **2026-02-05** |
| Fecha de fixing/vencimiento | **2026-03-06** |
| Dirección | **buy** (largo USD — cliente compra USD forward) |
| Liquidación | En USD: se intercambia la diferencia `(Fixing - Strike)` en USD |

**Mecánica:** Al vencimiento (2026-03-06) la TRM BanRep fija el rate. Si TRM > 3,717.75 el cliente gana; si TRM < 3,717.75 el cliente pierde.

**Fórmula:**
$$F(T) = \text{Spot} \times \frac{DF_{USD}(T)}{DF_{COP}(T)}$$
$$NPV_{COP} = \text{sign} \times N_{USD} \times (F(T) - K) \times DF_{COP}(T)$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'), override=True)

import QuantLib as ql
import pandas as pd
from datetime import datetime

from pricing.data.market_data import MarketDataLoader
from pricing.curves.curve_manager import CurveManager
from pricing.instruments.ndf import NdfPricer

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

loader = MarketDataLoader()
print('MarketDataLoader OK')

---
## 1. Constantes del Trade

In [ ]:
# ── Trade constants ───────────────────────────────────────────────────────────
NOTIONAL_USD   = 500_000
STRIKE         = 3_717.75      # Forward rate USD/COP pactado en inception
DIRECTION      = 'buy'         # Cliente largo USD
DATE_INCEPTION = '2026-02-05'  # Fecha de firma del contrato
DATE_MATURITY  = '2026-03-06'  # Fecha de fixing (TRM BanRep)
DATE_HOY       = '2026-03-03'  # Fecha de valoración

T_INCEPTION  = ql.Date(5,  2, 2026)
T_MATURITY   = ql.Date(6,  3, 2026)
T_HOY        = ql.Date(3,  3, 2026)

print(f'Nocional USD:   {NOTIONAL_USD:>12,.0f}')
print(f'Strike:         {STRIKE:>12,.2f}  USD/COP')
print(f'Dirección:      {DIRECTION}')
print(f'Inception:      {DATE_INCEPTION}')
print(f'Fixing/vto:     {DATE_MATURITY}')
print(f'Valoración:     {DATE_HOY}')

dias_totales  = T_MATURITY - T_INCEPTION
dias_restantes = T_MATURITY - T_HOY
print(f'\nPlazo total:    {dias_totales} días calendario')
print(f'Días restantes: {dias_restantes} días (al {DATE_HOY})')

---
## 2. Curvas a Inception (2026-02-05)

En inception el NDF se pacta **at-market**: el strike es igual al forward implícito de las curvas.
El NPV en inception debe ser ~0 (cualquier residual refleja bid-offer o interpolación).

In [ ]:
def build_cm(fecha_str: str) -> CurveManager:
    """Construye un CurveManager con las marcas de una fecha específica."""
    ibr     = loader.fetch_ibr_quotes(target_date=fecha_str)
    sofr_df = loader.fetch_sofr_curve(target_date=fecha_str)
    fx      = loader.fetch_usdcop_spot(target_date=fecha_str)

    d = datetime.strptime(fecha_str, '%Y-%m-%d')
    val_date = ql.Date(d.day, d.month, d.year)

    cm = CurveManager(valuation_date=val_date)
    cm.build_ibr_curve(ibr)
    cm.build_sofr_curve(sofr_df)
    cm.set_fx_spot(fx)
    return cm


print('=== Cargando marcas de inception: 2026-02-05 ===')
cm_inc = build_cm(DATE_INCEPTION)
print(f'FX spot:             {cm_inc.fx_spot:.2f}')
print(f'IBR curve ref date:  {cm_inc.ibr_curve.referenceDate()}')
print(f'SOFR curve ref date: {cm_inc.sofr_curve.referenceDate()}')

### 2.1 Nodos de las curvas a inception

In [ ]:
print('Nodos IBR (tasas par que entran al bootstrap):')
print(f'  {"Tenor":<12}  {"Tipo":<10}  {"Tasa (%)"}')
print('  ' + '-'*35)
for k, sq in cm_inc.ibr_quotes.items():
    tipo = 'Depósito' if k in ('ibr_1d','ibr_1m','ibr_3m','ibr_6m','ibr_12m') else 'OIS Swap'
    print(f'  {k:<12}  {tipo:<10}  {sq.value()*100:.4f}%')

print()
print('Nodos SOFR (tasas par):')
print(f'  {"Tenor":>6}  {"Tipo":<10}  {"Tasa (%)"}')
print('  ' + '-'*30)
for months, sq in cm_inc.sofr_quotes.items():
    tipo  = 'Depósito' if months <= 12 else 'OIS Swap'
    label = f'{months}M' if months < 12 else f'{months//12}Y'
    print(f'  {label:>6}  {tipo:<10}  {sq.value()*100:.4f}%')

---
## 3. La Mecánica del NDF — paso a paso

### 3.1 Forward implícito por paridad de tasas de interés

$$F(T) = \text{Spot} \times \frac{DF_{USD}(T)}{DF_{COP}(T)}$$

**Intuición:**  
- Un inversionista puede: (a) invertir USD a SOFR hasta T, o (b) convertir a COP hoy, invertir a IBR hasta T, y convertir de vuelta.  
- En equilibrio ambas opciones son iguales → el forward queda determinado por el ratio de discount factors.
- IBR > SOFR → el COP descuenta más fuerte → $DF_{COP}$ cae más rápido → F > Spot (prima forward en COP).

### 3.2 NPV del NDF

$$NPV_{COP} = \text{sign} \times N_{USD} \times (F(T) - K) \times DF_{COP}(T)$$

donde `sign = +1` para **buy** (largo USD), `sign = -1` para **sell**.  

**El `DF_COP` descuenta el payoff porque el NDF liquida en USD, pero convención local es valuar en COP.**

### 3.3 Delta (sensibilidad al spot)

$$\Delta_{COP} = \text{sign} \times N_{USD} \times DF_{COP}(T)$$

Interpretación: si el spot sube 1 COP/USD, el NPV cambia en `Δ_COP` pesos.

In [ ]:
DC = ql.Actual360()

# Discount factors a vencimiento (inception)
df_usd_inc = cm_inc.sofr_handle.discount(T_MATURITY)
df_cop_inc = cm_inc.ibr_handle.discount(T_MATURITY)
spot_inc   = cm_inc.fx_spot

# Forward implícito inception
fwd_inc = spot_inc * df_usd_inc / df_cop_inc

# Tiempo al vencimiento (ACT/360)
tau = DC.yearFraction(T_INCEPTION, T_MATURITY)

print('=== Cálculo manual del forward implícito (inception) ===')
print()
print(f'  Spot (inception):    {spot_inc:>10,.2f}  USD/COP')
print(f'  DF_USD ({DATE_MATURITY}):  {df_usd_inc:>10.6f}  (SOFR)')
print(f'  DF_COP ({DATE_MATURITY}):  {df_cop_inc:>10.6f}  (IBR)')
print(f'  F = Spot × DF_USD / DF_COP')
print(f'    = {spot_inc:.2f} × {df_usd_inc:.6f} / {df_cop_inc:.6f}')
print(f'    = {fwd_inc:>10,.4f}  USD/COP')
print()
print(f'  Strike pactado:      {STRIKE:>10,.2f}  USD/COP')
print(f'  Forward − Strike:    {fwd_inc - STRIKE:>+10,.4f}  USD/COP')
print(f'  tau (ACT/360):       {tau:.6f}')
print()
print(f'  → Forward implícito {'~= ' if abs(fwd_inc - STRIKE) < 20 else '≠  '}Strike  (se espera ~0 NPV en inception)')

---
## 4. Valoración a Inception

In [ ]:
ndf_inc = NdfPricer(cm_inc)
res_inc = ndf_inc.price(
    notional_usd=NOTIONAL_USD,
    strike=STRIKE,
    maturity_date=T_MATURITY,
    direction=DIRECTION,
)

print('=== NDF INCEPTION (2026-02-05) — perspectiva cliente ===')
print()
print(f'  Spot:                   {res_inc["spot"]:>12,.2f}  USD/COP')
print(f'  DF_USD:                 {res_inc["df_usd"]:>12.6f}')
print(f'  DF_COP:                 {res_inc["df_cop"]:>12.6f}')
print(f'  Forward implícito:      {res_inc["forward"]:>12,.4f}  USD/COP')
print(f'  Strike:                 {res_inc["strike"]:>12,.2f}  USD/COP')
print(f'  Forward − Strike:       {res_inc["forward"] - res_inc["strike"]:>+12,.4f}  USD/COP')
print(f'  Forward points:         {res_inc["forward_points"]:>+12,.2f}  (vs spot)')
print()
print(f'  NPV (COP):              {res_inc["npv_cop"]:>15,.0f}')
print(f'  NPV (USD):              {res_inc["npv_usd"]:>15,.2f}')
print()
print(f'  Delta (COP/1 USD move): {res_inc["delta_cop"]:>15,.0f}')
print(f'  Nocional:               {res_inc["notional_usd"]:>15,.0f} USD')
print(f'  Dirección:              {res_inc["direction"]}')

---
## 5. Curvas de Hoy (2026-03-03) — Mid-life Pricing

El NDF faltan **3 días** para vencer (fixing: 2026-03-06). ¿Qué cambió?
- FX spot puede haberse movido
- Curvas IBR y SOFR actualizadas
- El tiempo restante es muy corto → el DF ≈ 1, el NPV refleja casi directamente `(F_hoy - Strike) × N`

In [ ]:
print('=== Cargando marcas de hoy: 2026-03-03 ===')
cm_hoy = build_cm(DATE_HOY)
print(f'FX spot:             {cm_hoy.fx_spot:.2f}')
print(f'IBR  curve ref date: {cm_hoy.ibr_curve.referenceDate()}')
print(f'SOFR curve ref date: {cm_hoy.sofr_curve.referenceDate()}')
print()
print(f'FX inception: {spot_inc:.2f}  →  FX hoy: {cm_hoy.fx_spot:.2f}  '
      f'({(cm_hoy.fx_spot/spot_inc - 1)*100:+.2f}%)')

### 5.1 Comparación de curvas: inception vs hoy

In [ ]:
# Evaluar en la fecha de vencimiento del NDF
check_dates = [
    ('1M',  ql.Date(5,  3, 2026)),
    ('Vto', T_MATURITY),
    ('3M',  ql.Date(5,  5, 2026)),
    ('6M',  ql.Date(5,  8, 2026)),
    ('12M', ql.Date(5,  2, 2027)),
]

rows = []
for label, d in check_dates:
    try:
        ibr_i  = cm_inc.ibr_handle.zeroRate(d, DC, ql.Continuous).rate()
        ibr_h  = cm_hoy.ibr_handle.zeroRate(d, DC, ql.Continuous).rate()
        sofr_i = cm_inc.sofr_handle.zeroRate(d, DC, ql.Continuous).rate()
        sofr_h = cm_hoy.sofr_handle.zeroRate(d, DC, ql.Continuous).rate()
        rows.append({
            'Tenor': label,
            'Fecha': str(d),
            'IBR inc%': round(ibr_i * 100, 4),
            'IBR hoy%': round(ibr_h * 100, 4),
            'IBR Δbps': round((ibr_h - ibr_i) * 10_000, 1),
            'SOFR inc%': round(sofr_i * 100, 4),
            'SOFR hoy%': round(sofr_h * 100, 4),
            'SOFR Δbps': round((sofr_h - sofr_i) * 10_000, 1),
        })
    except Exception as e:
        rows.append({'Tenor': label, 'Fecha': str(d), 'Error': str(e)})

pd.DataFrame(rows)

---
## 6. Valoración Hoy — Cálculo Manual + NdfPricer

In [ ]:
# ── Cálculo manual ─────────────────────────────────────────────────────────
df_usd_hoy = cm_hoy.sofr_handle.discount(T_MATURITY)
df_cop_hoy = cm_hoy.ibr_handle.discount(T_MATURITY)
spot_hoy   = cm_hoy.fx_spot

fwd_hoy    = spot_hoy * df_usd_hoy / df_cop_hoy
sign       = 1.0 if DIRECTION == 'buy' else -1.0
npv_cop    = sign * NOTIONAL_USD * (fwd_hoy - STRIKE) * df_cop_hoy
npv_usd    = npv_cop / spot_hoy
delta_cop  = sign * NOTIONAL_USD * df_cop_hoy

print('=== Cálculo manual — hoy 2026-03-03 ===')
print()
print(f'  Spot hoy:               {spot_hoy:>12,.2f}  USD/COP')
print(f'  DF_USD ({DATE_MATURITY}):  {df_usd_hoy:>12.6f}')
print(f'  DF_COP ({DATE_MATURITY}):  {df_cop_hoy:>12.6f}')
print(f'  Forward implícito:      {fwd_hoy:>12,.4f}  USD/COP')
print(f'  Strike:                 {STRIKE:>12,.2f}  USD/COP')
print(f'  Forward − Strike:       {fwd_hoy - STRIKE:>+12,.4f}  USD/COP')
print()
print(f'  NPV_COP = sign × N_USD × (F − K) × DF_COP')
print(f'          = {sign:+.0f} × {NOTIONAL_USD:,.0f} × {fwd_hoy - STRIKE:+,.4f} × {df_cop_hoy:.6f}')
print(f'          = {npv_cop:>15,.2f}  COP')
print()
print(f'  NPV (USD) = NPV_COP / spot  =  {npv_usd:>12,.2f}  USD')
print(f'  Δ_COP     = sign × N × DF_COP = {delta_cop:>12,.0f}  COP/USD')

In [ ]:
# ── Verificar con NdfPricer ────────────────────────────────────────────────
ndf_hoy = NdfPricer(cm_hoy)
res_hoy = ndf_hoy.price(
    notional_usd=NOTIONAL_USD,
    strike=STRIKE,
    maturity_date=T_MATURITY,
    direction=DIRECTION,
)

print('=== NdfPricer — resultado completo (hoy) ===')
print()
for k, v in res_hoy.items():
    if isinstance(v, float):
        print(f'  {k:<20}: {v:>15,.4f}')
    else:
        print(f'  {k:<20}: {v}')

print()
assert abs(res_hoy['npv_cop'] - npv_cop) < 1, 'Discrepancia en NPV_COP!'
print('✓  NdfPricer coincide con el cálculo manual.')

---
## 7. Resumen: Inception vs Hoy

In [ ]:
W = 16
i, h = res_inc, res_hoy

print('=' * 68)
print('RESUMEN NDF — FW-BOCS-05.02.2026')
print('=' * 68)
print(f'  {"":<24}  {"INCEPTION 05-feb":>{W}}  {"HOY 03-mar":>{W}}  {"DELTA":>12}')
print(f'  {"-"*68}')
print(f'  {"FX spot":<24}  {i["spot"]:>{W},.2f}  {h["spot"]:>{W},.2f}  {h["spot"]-i["spot"]:>+12,.2f}')
print(f'  {"DF_USD":<24}  {i["df_usd"]:>{W}.6f}  {h["df_usd"]:>{W}.6f}  {h["df_usd"]-i["df_usd"]:>+12.6f}')
print(f'  {"DF_COP":<24}  {i["df_cop"]:>{W}.6f}  {h["df_cop"]:>{W}.6f}  {h["df_cop"]-i["df_cop"]:>+12.6f}')
print(f'  {"Forward implícito":<24}  {i["forward"]:>{W},.4f}  {h["forward"]:>{W},.4f}  {h["forward"]-i["forward"]:>+12,.4f}')
print(f'  {"Strike":<24}  {i["strike"]:>{W},.2f}  {h["strike"]:>{W},.2f}')
print(f'  {"Forward − Strike":<24}  {i["forward"]-i["strike"]:>{W}+,.4f}  {h["forward"]-h["strike"]:>{W}+,.4f}')
print(f'  {"-"*68}')
print(f'  {"NPV (COP)":<24}  {i["npv_cop"]:>{W},.0f}  {h["npv_cop"]:>{W},.0f}  {h["npv_cop"]-i["npv_cop"]:>+12,.0f}')
print(f'  {"NPV (USD)":<24}  {i["npv_usd"]:>{W},.2f}  {h["npv_usd"]:>{W},.2f}  {h["npv_usd"]-i["npv_usd"]:>+12,.2f}')
print(f'  {"Delta (COP)":<24}  {i["delta_cop"]:>{W},.0f}  {h["delta_cop"]:>{W},.0f}')

---
## 8. Descomposición del P&L: FX vs Tasas

Usando el método `pnl_inception()` del `NdfPricer`:

- **P&L FX:** ¿Cuánto del cambio se explica porque el spot se movió?  
  → Repricia con el spot actual pero manteniendo el ratio DF_USD/DF_COP de inception.

- **P&L Rates:** ¿Cuánto se explica por el movimiento de curvas?  
  → Repricia con el spot de inception pero usando los DFs actuales.

- **Cross:** Interacción entre los dos efectos (generalmente pequeño para NDFs cortos).

In [ ]:
pnl = ndf_hoy.pnl_inception(
    notional_usd=NOTIONAL_USD,
    strike=STRIKE,
    maturity_date=T_MATURITY,
    direction=DIRECTION,
    fx_inception=spot_inc,   # spot exacto de inception
)

print('=== Descomposición P&L (inception → hoy) ===')
print()
print(f'  FX inception:          {pnl["fx_inception"]:>12,.2f}  USD/COP')
print(f'  FX hoy:                {pnl["spot_current"]:>12,.2f}  USD/COP')
print(f'  ΔFX:                   {pnl["spot_current"] - pnl["fx_inception"]:>+12,.2f}  USD/COP')
print()
print(f'  Forward inception:     {STRIKE:>12,.2f}  (= strike, at-market)')
print(f'  Forward rate-only:     {pnl["forward_rate_only"]:>12,.4f}  (solo curvas se mueven)')
print(f'  Forward FX-only:       {pnl["forward_fx_only"]:>12,.4f}  (solo spot se mueve)')
print(f'  Forward actual hoy:    {pnl["forward_current"]:>12,.4f}')
print()
print(f'  P&L FX    (COP):       {pnl["pnl_fx_cop"]:>+15,.0f}')
print(f'  P&L Rates (COP):       {pnl["pnl_rates_cop"]:>+15,.0f}')
print(f'  P&L Cross (COP):       {pnl["pnl_cross_cop"]:>+15,.0f}')
print(f'  P&L Total (COP):       {pnl["npv_cop"]:>+15,.0f}')
print()
total_check = pnl['pnl_fx_cop'] + pnl['pnl_rates_cop'] + pnl['pnl_cross_cop']
print(f'  Check (FX+Rates+Cross = Total): {total_check:>+15,.0f}  ✓' if abs(total_check - pnl['npv_cop']) < 1 else 'ERROR')

---
## 9. DV01 — Sensibilidad a tasas (+1 bp)

Para un NDF de plazo corto el DV01 es pequeño porque:
$$DV01 \approx N_{USD} \times \frac{\partial}{\partial r} \left[ \frac{DF_{USD}}{DF_{COP}} \right] \times DF_{COP} \times \Delta r$$

- IBR sube 1bp → $DF_{COP}$ baja → el forward sube menos de lo que baja el descuento → efecto neto negativo para buy.
- SOFR sube 1bp → $DF_{USD}$ baja → el forward baja → NPV disminuye para buy.

In [ ]:
dv01 = ndf_hoy.dv01(
    notional_usd=NOTIONAL_USD,
    strike=STRIKE,
    maturity_date=T_MATURITY,
    direction=DIRECTION,
    bump_bps=1.0,
)

print('=== DV01 (+1 bp paralelo) ===')
print()
print(f'  Base NPV (COP):        {dv01["base_npv_cop"]:>15,.0f}')
print()
print(f'  DV01 IBR  (COP/bp):    {dv01["dv01_ibr_cop"]:>+15,.0f}')
print(f'  DV01 SOFR (COP/bp):    {dv01["dv01_sofr_cop"]:>+15,.0f}')
print(f'  DV01 neto (COP/bp):    {dv01["dv01_total_cop"]:>+15,.0f}')
print()
spot_dv = cm_hoy.fx_spot
print(f'  DV01 IBR  (USD/bp):    {dv01["dv01_ibr_cop"]/spot_dv:>+12,.0f}')
print(f'  DV01 SOFR (USD/bp):    {dv01["dv01_sofr_cop"]/spot_dv:>+12,.0f}')
print(f'  DV01 neto (USD/bp):    {dv01["dv01_total_cop"]/spot_dv:>+12,.0f}')
print()
print('  Nota: DV01 pequeño para NDF de plazo corto (~1 mes).')
print('  La exposición dominante es el delta (sensibilidad al spot FX).')

---
## 10. Curva de Forwards Implícitos vs Mercado

Comparamos el forward implícito del modelo (paridad IBR/SOFR) contra las cotizaciones de mercado (`cop_fwd_points`). El `basis` indica si el mercado está pagando prima o descuento sobre el modelo.

In [ ]:
cop_fwd = loader.fetch_cop_forwards()

if not cop_fwd.empty:
    df_implied = ndf_hoy.implied_curve(cop_fwd)
    print('=== Forward implícito vs mercado (hoy) ===')
    print()
    print(df_implied.to_string(index=False))
    print()
    print('Basis = market − implied')
    print('  +basis → mercado cotiza más caro que el modelo (COP más fuerte de lo que dicen las tasas)')
    print('  -basis → mercado más barato que el modelo')
else:
    print('No hay datos de cop_fwd_points disponibles.')
    print('Usando solo el forward implícito del modelo.')
    print()
    print(f'  Forward implícito a vto ({DATE_MATURITY}): {res_hoy["forward"]:,.4f} USD/COP')
    print(f'  Strike:                                 {STRIKE:,.2f} USD/COP')
    print(f'  Basis (forward − strike):               {res_hoy["forward"] - STRIKE:+,.4f}')

---
## 11. Resumen Ejecutivo

In [ ]:
print('═' * 60)
print('NDF FW-BOCS-05.02.2026 — RESUMEN EJECUTIVO')
print('Valoración:', DATE_HOY, '| Vencimiento:', DATE_MATURITY, f'(en {T_MATURITY - T_HOY} días)')
print('═' * 60)
print()
print(f'  POSICIÓN')
print(f'  Nocional:     USD {NOTIONAL_USD:>10,.0f}')
print(f'  Strike:       COP {STRIKE:>10,.2f}')
print(f'  Dirección:        {DIRECTION.upper()} (largo USD)')
print()
print(f'  MERCADO HOY')
print(f'  Spot:         COP {cm_hoy.fx_spot:>10,.2f}')
print(f'  Forward imp:  COP {res_hoy["forward"]:>10,.4f}')
print(f'  Fwd points:   COP {res_hoy["forward_points"]:>+10,.2f}  (vs spot)')
print()
print(f'  VALORACIÓN (modelo — curvas IBR/SOFR)')
print(f'  NPV (COP):    {res_hoy["npv_cop"]:>15,.0f}')
print(f'  NPV (USD):    {res_hoy["npv_usd"]:>15,.2f}')
print()
print(f'  RIESGO')
print(f'  Delta (COP):  {res_hoy["delta_cop"]:>15,.0f}  por 1 USD/COP move')
print(f'  DV01 IBR:     {dv01["dv01_ibr_cop"]:>+15,.0f}  COP / bp')
print(f'  DV01 SOFR:    {dv01["dv01_sofr_cop"]:>+15,.0f}  COP / bp')
print()
print(f'  P&L DESDE INCEPTION')
print(f'  P&L total:    {res_hoy["npv_cop"] - res_inc["npv_cop"]:>+15,.0f}  COP')
print(f'  del cual FX:  {pnl["pnl_fx_cop"]:>+15,.0f}  COP')
print(f'  del cual ΔR:  {pnl["pnl_rates_cop"]:>+15,.0f}  COP')
print('═' * 60)